In [ ]:
import csv
import random
import time
from datetime import datetime
from collections import deque

# ==========================================
# TASK 1: LINEAR SEARCH AND BINARY SEARCH
# ==========================================

def load_orders(filename):
    orders = []
    with open(filename, mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            orders.append({
                "orderId": row["orderId"].strip(),
                "customerId": row["customerId"].strip(),
                "amount": float(row["amount"]),
                "createdAt": row["createdAt"].strip()
            })
    return orders

orders = load_orders("orders.csv")

def linear_search(orders, order_id):
    for order in orders:
        if order["orderId"] == order_id:
            return order
    return None


def binary_search(orders, order_id):
    low = 0
    high = len(orders) - 1

    while low <= high:
        mid = (low + high) // 2

        if orders[mid]["orderId"] == order_id:
            return orders[mid]

        elif orders[mid]["orderId"] < order_id:
            low = mid + 1

        else:
            high = mid - 1

    return None


order_id = "ORD-045000"

start = time.perf_counter()
result1 = linear_search(orders, order_id)
linear_time = time.perf_counter() - start

start = time.perf_counter()
result2 = binary_search(orders, order_id)
binary_time = time.perf_counter() - start

print("\nTASK 1: SEARCHING")
print("Linear Search:", result1)
print("Linear Search Time:", linear_time)

print("Binary Search:", result2)
print("Binary Search Time:", binary_time)


# ==========================================
# TASK 2: SORTING ALGORITHMS
# ==========================================

def bubble_sort(data):
    arr = data.copy()
    n = len(arr)

    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j]["amount"] < arr[j + 1]["amount"]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]

    return arr


def merge_sort(arr):
    if len(arr) <= 1:
        return arr

    mid = len(arr) // 2

    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])

    return merge(left, right)


def merge(left, right):
    result = []
    i = 0
    j = 0

    while i < len(left) and j < len(right):
        if left[i]["amount"] >= right[j]["amount"]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1

    result.extend(left[i:])
    result.extend(right[j:])

    return result


sample = orders[:10000]

start = time.perf_counter()
bubble_result = bubble_sort(sample)
bubble_time = time.perf_counter() - start

start = time.perf_counter()
merge_result = merge_sort(sample)
merge_time = time.perf_counter() - start

start = time.perf_counter()
builtin_result = sorted(
    sample,
    key=lambda x: x["amount"],
    reverse=True
)
builtin_time = time.perf_counter() - start

print("\nTASK 2: SORTING")
print("Bubble Sort Time:", bubble_time)
print("Merge Sort Time:", merge_time)
print("Built-in Sort Time:", builtin_time)

print("\nTop 5 Orders by Amount:")
for order in builtin_result[:5]:
    print(order)


# ==========================================
# TASK 3: DUPLICATE ORDER DETECTION
# ==========================================

def detect_duplicates(orders):
    seen = {}
    duplicates = []

    for order in orders:
        created = datetime.strptime(
            order["createdAt"],
            "%Y-%m-%d %H:%M:%S"
        )

        minute = created.strftime("%Y-%m-%d %H:%M")

        key = (
            order["customerId"],
            order["amount"],
            minute
        )

        if key in seen:
            duplicates.append(order)
        else:
            seen[key] = order

    return duplicates


duplicates = detect_duplicates(orders)

print("\nTASK 3: DUPLICATE DETECTION")
print("Number of Duplicate Orders:", len(duplicates))

for order in duplicates[:5]:
    print(order)


# ==========================================
# TASK 4: STACK AND QUEUE
# ==========================================

class Stack:
    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        if not self.items:
            return None
        return self.items.pop()

    def peek(self):
        if not self.items:
            return None
        return self.items[-1]


class Queue:
    def __init__(self):
        self.items = []

    def enqueue(self, item):
        self.items.append(item)

    def dequeue(self):
        if not self.items:
            return None
        return self.items.pop(0)


# Custom Stack for undo operation

undo_stack = Stack()

undo_stack.push("Order 100 edited")
undo_stack.push("Order 200 edited")
undo_stack.push("Order 300 edited")

print("\nTASK 4: STACK")
print("Undo Last Edit:", undo_stack.pop())


# Custom Queue for dispatch

dispatch_queue = Queue()

for order in orders[:5]:
    dispatch_queue.enqueue(order["orderId"])

print("\nCUSTOM QUEUE")
print("Dispatch Order:", dispatch_queue.dequeue())


# Built-in equivalents

builtin_stack = []

builtin_stack.append("Order 100 edited")
builtin_stack.append("Order 200 edited")
builtin_stack.append("Order 300 edited")

print("\nBUILT-IN STACK")
print("Undo Last Edit:", builtin_stack.pop())


builtin_queue = deque()

for order in orders[:5]:
    builtin_queue.append(order["orderId"])

print("\nBUILT-IN QUEUE")
print("Dispatch Order:", builtin_queue.popleft())


# ==========================================
# TASK 5: GREEDY AND DYNAMIC PROGRAMMING
# ==========================================

def greedy_change(amount, denominations):
    result = {}

    for coin in denominations:
        count = amount // coin

        if count > 0:
            result[coin] = count
            amount %= coin

    if amount != 0:
        return None

    return result


def dp_change(amount, denominations):
    dp = [float("inf")] * (amount + 1)
    dp[0] = 0

    for i in range(1, amount + 1):
        for coin in denominations:
            if coin <= i:
                dp[i] = min(dp[i], dp[i - coin] + 1)

    if dp[amount] == float("inf"):
        return None

    result = {}
    remaining = amount

    while remaining > 0:
        for coin in denominations:
            if coin <= remaining and dp[remaining] == dp[remaining - coin] + 1:
                result[coin] = result.get(coin, 0) + 1
                remaining -= coin
                break

    return result


# Greedy example using Indian denominations

amount = 289
denominations = [200, 100, 50, 20, 10, 5, 2, 1]

greedy_result = greedy_change(amount, denominations)

print("\nTASK 5: GREEDY APPROACH")
print("Refund Amount:", amount)
print("Greedy Result:", greedy_result)
print("Total Notes/Coins:", sum(greedy_result.values()))


# Example where greedy fails

amount = 6
denominations = [4, 3, 1]

greedy_result = greedy_change(amount, denominations)
dp_result = dp_change(amount, denominations)

print("\nGREEDY FAILURE EXAMPLE")
print("Amount:", amount)
print("Denominations:", denominations)
print("Greedy Result:", greedy_result)
print("Greedy Coin Count:", sum(greedy_result.values()))

print("\nDYNAMIC PROGRAMMING")
print("DP Result:", dp_result)
print("DP Minimum Coins:", sum(dp_result.values()))


# ==========================================
# FINAL SUMMARY
# ==========================================

print("\nALL 5 TASKS COMPLETED SUCCESSFULLY")

RETAIL ORDER DESK
Total orders loaded: 50000

TASK 1 - SEARCH
Target order: O49999
Linear search result: None
Linear search time: 0.00408400 seconds
Binary search result: None
Binary search time: 0.00002910 seconds
Note: Binary search requires the data to be sorted by orderId first.

SELECTION SORT TRACE - DESCENDING
Initial: [450, 120, 700, 300, 900]
After pass 1: [900, 120, 700, 300, 450]
After pass 2: [900, 700, 120, 300, 450]
After pass 3: [900, 700, 450, 300, 120]

TASK 2 - SORTING (10,000 records)
Selection sort (O(n^2)): 10.797933 seconds
Merge sort (O(n log n)): 0.061929 seconds
Python built-in sorted(): 0.002705 seconds
Selection sort matches built-in: True
Merge sort matches built-in: True
Top 5 orders by amount:
ORD-008263 Rs. 2499.55
ORD-004953 Rs. 2498.60
ORD-000367 Rs. 2498.44
ORD-002800 Rs. 2498.16
ORD-007781 Rs. 2497.95

TASK 3 - DUPLICATE DETECTION
Duplicate pairs found: 0
Average time complexity: O(n)
Space complexity: O(n)

TASK 4 - STACK AND QUEUE
Custom stack - und